In [ ]:
!pip install --upgrade pip

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install -U typing_extensions

import torch; torch._dynamo.config.recompile_limit = 64;


In [ ]:
%%capture
!pip install --no-deps --upgrade timm # Only for Gemma 3N

In [ ]:
!pip install opencv-python

In [ ]:
from huggingface_hub import login
import os

# Paste your token here (get it from: https://huggingface.co/settings/tokens)
HF_TOKEN = "" 

login(token=HF_TOKEN)

In [ ]:
from unsloth import FastVisionModel
import torch

model, processor = FastVisionModel.from_pretrained(
    "blind-assist/gemma-3n-4b-finetune-8500",
    load_in_4bit = False,
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
!pip uninstall torchcodec -y

In [ ]:
from datasets import load_dataset, Video, Dataset
from huggingface_hub import hf_hub_download
from itertools import islice
import os

# Load test dataset
dataset_stream = load_dataset(
    "blind-assist/walk",
    split="test",
    streaming=True
)

dataset_stream = dataset_stream.cast_column("video", Video(decode=False))

# Download first 50 videos
SAVE_DIR = "test_videos"
os.makedirs(SAVE_DIR, exist_ok=True)

N = 50
local_data = []

for item in islice(dataset_stream, N):
    video_hf_path = item["video"]["path"]
    filename = video_hf_path.split("/")[-1]
    local_path = os.path.join(SAVE_DIR, "test", filename)

    if not os.path.exists(local_path):
        hf_hub_download(
            repo_id="blind-assist/walk",
            filename=f"test/{filename}",
            repo_type="dataset",
            local_dir=SAVE_DIR,
            local_dir_use_symlinks=False
        )

    item["video"]["path"] = local_path
    local_data.append(item)

test_dataset = Dataset.from_list(local_data)
print(f"Loaded {len(test_dataset)} test samples")

In [ ]:
import cv2
import numpy as np
from PIL import Image

def downsample_video(video_path, num_frames=2):
    """Extracts evenly spaced frames for VLM context."""
    
    vidcap = cv2.VideoCapture(video_path)
    total_frames = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0: return []
    
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []
    for i in indices:
        vidcap.set(cv2.CAP_PROP_POS_FRAMES, i)
        success, image = vidcap.read()
        if success:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(image))
    vidcap.release()
    return frames


In [ ]:
from unsloth import get_chat_template

processor = get_chat_template(
    processor,
    "gemma-3n"
)

In [ ]:
import time
import csv
from transformers import TextStreamer

# Enable Unsloth optimized inference
FastVisionModel.for_inference(model)

# Output CSV file
output_csv = "test_inference_results.csv"

print(f"Processing {len(test_dataset)} test videos")

# Instruction for the model
instruction = ("Given the visual input from the user's forward perspective, "
               "generate exactly one short sentence to guide a visually impaired user "
               "by identifying critical obstacles or landmarks, describing their locations "
               "using clock directions relative to the user (12 o'clock is straight ahead), "
               "including relevant details such as size, material, or distance, and giving "
               "one clear action, while prioritizing immediate safety and avoiding any extra explanation.")

# Prepare results
results = []

# Process each video
for idx in range(len(test_dataset)):
    sample = test_dataset[idx]
    video_path = sample["video"]["path"]
    ground_truth = sample["alter"]
    video_filename = video_path.split("/")[-1]
    
    print(f"\n[{idx+1}/{len(test_dataset)}] Processing: {video_filename}")
    
    try:
        # Extract frames from video
        frames = downsample_video(video_path)
        
        # Construct the multimodal message
        messages = [{"role": "user", "content": [{"type": "text", "text": instruction}]}]
        
        # Interleave frames into message content
        for img in frames:
            messages[0]["content"].append({"type": "image", "image": img})
        
        # Apply chat template and process inputs
        input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(
            images=frames,
            text=input_text,
            add_special_tokens=False,
            return_tensors="pt",
        ).to("cuda")

        start_time = time.time()

        # Generate output
        output = model.generate(
            **inputs,
            max_new_tokens=128,
            use_cache=True,
            temperature=1.0,
            top_p=0.95,
            top_k=64
        )

        end_time = time.time()
        inference_time = round(end_time - start_time, 3)
        
        # Decode the output
        generated_text = processor.decode(output[0], skip_special_tokens=True)
        
        # Extract only the generated response (remove the prompt)
        if "model" in generated_text:
            prediction = generated_text.split("model")[-1].strip()
        elif "assistant" in generated_text.lower():
            prediction = generated_text.split("assistant")[-1].strip()
        else:
            prediction = generated_text.strip()
        
        # Remove any remaining newlines and extra whitespace
        prediction = " ".join(prediction.split())
        
        print(f"Prediction: {prediction}")
        print(f"Ground Truth: {ground_truth}")
        
        # Store result
        results.append({
            "video_id": idx + 1,
            "video_filename": video_filename,
            "prediction": prediction,
            "ground_truth": ground_truth,
            "inference_time_s": inference_time
        })
        
    except Exception as e:
        print(f"Error processing {video_filename}: {str(e)}")
        results.append({
            "video_id": idx + 1,
            "video_filename": video_filename,
            "prediction": f"ERROR: {str(e)}",
            "ground_truth": ground_truth,
            "inference_time_s": 0.0
        })

# Save results to CSV
print(f"\nSaving results to {output_csv}...")
with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['video_id', 'video_filename', 'prediction', 'ground_truth', 'inference_time_s']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    writer.writeheader()
    writer.writerows(results)

print(f"\nProcessing complete! Results saved to {output_csv}")
print(f"Successfully processed {len([r for r in results if not r['prediction'].startswith('ERROR')])} out of {len(test_dataset)} videos")



In [ ]:
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
# Display results preview
import pandas as pd
df = pd.read_csv(output_csv)
print(df.head(10))
print(f"\nAverage inference time: {df['inference_time_s'].mean():.3f}s")